In [1]:
import pandas as pd
import pysam
from pyfaidx import Fasta

from crop_embed.data.coords import (
    FASTA_PATH,
    FLANKING_PATH,
    chrom_name_map,
)

REMAPPED_VCF_PATH = "../rice_data/sativas413_msu7.vcf"

ref          = Fasta(FASTA_PATH)
chrom_names  = chrom_name_map(FASTA_PATH)
flanking_seq = pd.read_csv(FLANKING_PATH, sep="\t").set_index("snp_id")

print("Chromosomes:", chrom_names)
print(f"Flanking table: {len(flanking_seq):,} SNPs")

Chromosomes: {1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9', 10: '10', 11: '11', 12: '12'}
Flanking table: 42,755 SNPs


In [2]:
# --- REF allele check ---
# CrossMap writes standard 1-based VCF POS; pysam returns the raw value
# without subtracting 1, so we use rec.pos - 1 as the 0-based FASTA index.

ref_match    = []
ref_mismatch = []

vcf = pysam.VariantFile(REMAPPED_VCF_PATH)
for rec in vcf.fetch():
    chrom_int  = int(rec.chrom) if rec.chrom.isdigit() else int(rec.chrom.lstrip("chr"))
    seq_key    = chrom_names[chrom_int]
    pos0       = rec.pos - 1   # 0-based FASTA index
    fasta_base = ref[seq_key][pos0 : pos0 + len(rec.ref)].seq.upper()
    vcf_ref    = rec.ref.upper()

    entry = {
        "snp_id":     rec.id,
        "chr":        chrom_int,
        "pos":        rec.pos,
        "vcf_ref":    vcf_ref,
        "fasta_base": fasta_base,
    }
    (ref_match if fasta_base == vcf_ref else ref_mismatch).append(entry)

vcf.close()

n = len(ref_match) + len(ref_mismatch)
print(f"REF matches FASTA : {len(ref_match):,} / {n:,}  ({100 * len(ref_match) / n:.2f}%)")
print(f"REF mismatches    : {len(ref_mismatch):,}")
if ref_mismatch:
    print(pd.DataFrame(ref_mismatch).head(10))

REF matches FASTA : 36,900 / 36,900  (100.00%)
REF mismatches    : 0


In [3]:
def match_exclude_n(s1, s2):
    return all(x1 == x2 or x1 == "N" or x2 == "N" for x1, x2 in zip(s1, s2))

In [4]:
# --- Flanking-sequence check ---
HALF_WINDOW = 16
results = []

vcf = pysam.VariantFile(REMAPPED_VCF_PATH)
for rec in vcf.fetch():
    snp_id = rec.id
    if snp_id not in flanking_seq.index:
        continue

    chrom_int = int(rec.chrom) if rec.chrom.isdigit() else int(rec.chrom.lstrip("chr"))
    seq_key   = chrom_names[chrom_int]
    pos0      = rec.pos - 1   # 0-based FASTA index

    start  = max(0, pos0 - HALF_WINDOW)
    window = ref[seq_key][start : pos0 + HALF_WINDOW + 1].seq.upper()

    if len(window) != 2 * HALF_WINDOW + 1:
        continue  # near chromosome boundary

    stored = flanking_seq.loc[snp_id]
    results.append({
        "snp_id":      snp_id,
        "chr":         chrom_int,
        "pos_msu7":    rec.pos,
        "left_match":  window[:HALF_WINDOW]   == stored["X5p_MSU6"].upper(),
        "right_match": window[HALF_WINDOW+1:] == stored["X3p_MSU6"].upper(),
        "match_excl_N": match_exclude_n(window[:HALF_WINDOW], stored["X5p_MSU6"].upper()) and
                        match_exclude_n(window[HALF_WINDOW+1:], stored["X3p_MSU6"].upper()),
    })

vcf.close()
results_df = pd.DataFrame(results)
print(f"SNPs checked: {len(results_df):,}")

SNPs checked: 35,745


In [5]:
both_match = results_df["left_match"] & results_df["right_match"]
n          = len(results_df)

print(f"Both flanks match : {both_match.sum():,} / {n:,}  ({100 * both_match.mean():.2f}%)")
print(f"Both match exclude N : {results_df['match_excl_N'].sum():,} / {n:,}  ({100 * results_df['match_excl_N'].mean():.2f}%)")
print(f"Left flank only   : {(results_df['left_match'] & ~results_df['right_match']).sum():,}")
print(f"Right flank only  : {(~results_df['left_match'] & results_df['right_match']).sum():,}")
print(f"Neither           : {(~results_df['left_match'] & ~results_df['right_match']).sum():,}")

Both flanks match : 35,683 / 35,745  (99.83%)
Both match exclude N : 35,683 / 35,745  (99.83%)
Left flank only   : 0
Right flank only  : 0
Neither           : 62


In [6]:
results_df.groupby("chr").apply(
    lambda g: pd.Series({
        "n_snps":     len(g),
        "both_match": (g["left_match"] & g["right_match"]).sum(),
        "pct_match":  f"{100 * (g['left_match'] & g['right_match']).mean():.1f}%",
    }),
    include_groups=False,
)

,n_snps,both_match,pct_match
chr,,,
1,6235,6235,100.0%
2,3757,3757,100.0%
3,4210,4203,99.8%
4,2783,2776,99.7%
5,2723,2712,99.6%
6,3134,3134,100.0%
7,2015,2015,100.0%
8,2217,2217,100.0%
9,1922,1922,100.0%
